# Batch projection summaries for P-ADO transition outputs

This notebook scans multiple ordinary P-ADO output files (`*_all.csv` or `*_all.csv.gz`), computes one-dimensional projection summaries under constant sigma/I gates and optional Gaussian sigma/I weights, and writes combined summary tables.

This notebook is intended for ordinary all-output files. It is not intended for singular diagnostic files or regular files produced after removing singular/near-singular Jacobian points.

The regular file is generated after removing singular or near-singular Jacobian points. Therefore, the delta-sigma/I and AT-sigma/I points may no longer form a complete rectangular grid. The grid-based routines in `YAnalysis.py` are not designed as the default analysis method for regular files.

For regular files, users should treat the data as a sparse point set rather than a complete grid. If regular-file analysis is needed, use dedicated sparse-data methods, scatter/table diagnostics, or start from the corresponding `*_all.csv.gz` file and apply an explicit regular-point mask. Do not directly feed regular files into ordinary grid-based notebooks unless the missing-grid structure is handled explicitly.

This notebook writes two summary tables:

- `YAll_transition_summary_delta_wide.csv`: a wide table that keeps most available columns for downstream analysis.
- `YAll_transition_summary_delta_view.csv`: a compact view table with selected columns for quick reading and spreadsheet inspection.


## Imports

In [ ]:
from pathlib import Path

import pandas as pd

from YAnalysis import summarize_transition

## User configuration

In [ ]:
# DATA_DIR: folder containing generated P-ADO output CSV/CSV.GZ files.
# Replace this placeholder with the directory containing generated P-ADO CSV or CSV.GZ files.
DATA_DIR = Path(r"path/to/csv_directory")

# OUTPUT_DIR: folder for saving combined summary tables.
OUTPUT_DIR = Path(r"path/to/output_directory")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# CSV_PATTERN: file-name pattern used to select ordinary all-output files.
# For the ordinary projection-summary workflow, use *_all.csv.gz or *_all.csv files.
# Do not use *_detJ_regular.csv.gz or *_detJ_singular.csv.gz as the default input for this notebook.
CSV_PATTERN = "*_all.csv.gz"

# For plain CSV outputs, users may use:
# CSV_PATTERN = "*_all.csv"


## Batch scan ordinary all-output CSV files

The default scan uses `CSV_PATTERN` to find ordinary all-output files in `DATA_DIR`. For a single file, users may alternatively replace the scan with a one-item list such as `csv_paths = [Path(r"path/to/file_name_all.csv.gz")]`.


In [ ]:
csv_paths = sorted(DATA_DIR.glob(CSV_PATTERN))

# For a single file, users may alternatively comment out the scan above and use:
# csv_paths = [Path(r"path/to/file_name_all.csv.gz")]

csv_paths[:5], len(csv_paths)


## Define constant gates and Gaussian weights

`SIGMA_GATES` defines constant sigma/I integration windows. `GAUSSIAN_WEIGHTS` defines optional Gaussian weights centered at selected sigma/I values.


In [ ]:
# SIGMA_GATES: constant sigma/I integration windows.
# "all": None means use the full available sigma/I range.
SIGMA_GATES = {
    "all": None,
    "s0.1": (0.05, 0.15),
    "s0.2": (0.15, 0.25),
    "s0.3": (0.25, 0.35),
    "s0.4": (0.35, 0.45),
    "s0.5": (0.45, 0.55),
}

# GAUSSIAN_WEIGHTS: Gaussian weights centered at selected sigma/I values.
# gaus_mu: center of the Gaussian weight in sigma/I.
# gaus_sigma: standard deviation of the Gaussian weight.
GAUSSIAN_WEIGHTS = [
    {"label": "gauss_s0.1_sig0.02", "gaus_mu": 0.10, "gaus_sigma": 0.02, "range": None},
    {"label": "gauss_s0.2_sig0.02", "gaus_mu": 0.20, "gaus_sigma": 0.02, "range": None},
    {"label": "gauss_s0.3_sig0.02", "gaus_mu": 0.30, "gaus_sigma": 0.02, "range": None},
    {"label": "gauss_s0.4_sig0.02", "gaus_mu": 0.40, "gaus_sigma": 0.02, "range": None},
    {"label": "gauss_s0.5_sig0.02", "gaus_mu": 0.50, "gaus_sigma": 0.02, "range": None},
]


## Compute summaries

Each selected transition file is summarized with the configured constant gates and Gaussian weights. `hdi_mass=0.68` requests a 68% highest-density interval mass.


In [ ]:
rows = []
for csv_path in csv_paths:
    try:
        df_one = summarize_transition(
            csv_path,
            sigma_gates=SIGMA_GATES,
            gaussian_weights=GAUSSIAN_WEIGHTS,
            hdi_mass=0.68,  # 68% highest-density interval mass.
        )
        rows.append(df_one)
    except Exception as exc:
        rows.append(pd.DataFrame([{
            "transition": csv_path.stem,
            "error": repr(exc),
        }]))

summary_df = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

display_cols = [
    "transition",
    "quantity",
    "gate_label",
    "weight_kind",
    "gate_type",
    "gate_range",
    "gaus_mu",
    "gaus_sigma",
    "mean",
    "std",
    "eq_q16",
    "eq_q50",
    "eq_q84",
    "eq_err_minus",
    "eq_err_plus",
    "hdi_interval_count",
    "mode",
    "hdi_interval",
    "hdi_interval_mass",
    "hdi_interval_mass_fraction",
    "hdi_interval_mass_rank",
]
available_display_cols = [col for col in display_cols if col in summary_df.columns]

# summary_df[available_display_cols]: compact view table for quick inspection.
summary_df[available_display_cols].head(20)


## Save summary CSV tables

The wide table keeps most available columns for downstream analysis. The view table keeps selected columns for quick reading and spreadsheet inspection.


In [ ]:
wide_path = OUTPUT_DIR / "YAll_transition_summary_delta_wide.csv"
view_path = OUTPUT_DIR / "YAll_transition_summary_delta_view.csv"

# summary_wide_df: complete output table for downstream analysis.
summary_wide_df = summary_df.drop(
    columns=["hdi_text", "summary_text"],
    errors="ignore",
)
summary_wide_df.to_csv(wide_path, index=False)

# The view table keeps selected columns for quick reading and spreadsheet inspection.
summary_df[available_display_cols].to_csv(view_path, index=False)

wide_path, view_path


## Optional: inspect multi-interval HDI rows

`multi_hdi` contains rows with more than one HDI interval. These rows are useful for identifying multi-modal or multi-solution cases that may need closer inspection.


In [ ]:
# multi_hdi: rows with more than one HDI interval, useful for identifying multi-modal or multi-solution cases.
if "hdi_interval_count" in summary_df.columns:
    multi_hdi = summary_df[summary_df["hdi_interval_count"].fillna(0) > 1]
else:
    multi_hdi = pd.DataFrame()

multi_hdi
